# Assignment 5: Autoencoders
Date: 

Report produced by:  
COMP 3705 Group 3

## Data Preprocessing

### Data Loading

In [ ]:
import pandas as pd
import numpy as np
import math as m
import tensorflow as tf
from sklearn.model_selection import train_test_split as tts
from keras.models import Model, Sequential
from keras.layers import Input, Dense
import matplotlib.pyplot as plt
# Set TensorFlow deprication warnings to False
import tensorflow.python.util.deprecation as deprecation
deprecation._PRINT_DEPRECATION_WARNINGS = False

# Load the Dataset and Check Shape
df = pd.read_csv("csvImages 10k x 784.csv")
df_clean = df.copy()
print("Data Shape:", df_clean.shape)

### Check Data Integrity

In [ ]:
print("Number of Duplicate Rows:",df_clean.duplicated().sum())
print("Min value across all columns:",df_clean.min().min())
print("Max value across all columns:",df_clean.max().max())
print("Number of missing values in data:",df_clean.isna().sum().sum())

As the data across all columns ranges between 0 and 255, we can determine that there are no unexpected RGB values.  
Additionally, since there are also no missing values anywhere in the data we can proceed with the dataset as-is.

In [ ]:
df_clean = df_clean.drop_duplicates()
print("Cleaned Data Shape:", df_clean.shape)

### Reshape for Visualizations

In [ ]:
# Convert DataFrame to Numpy Array and Normalize Values
img_data = df_clean.values/255
img_data = img_data.reshape((-1,28,28))
print("Shape of Array:",img_data.shape)
print("Shape of Each Row:",img_data[0].shape)

After checking the data, we converted the DataFrame into a NumPy array and normalized the pixel values by dividing them by 255. This changed the value range from 0–255 to  0–1. Normalization was done to helps the neural network train more effectively by ensuring no one pixel disproportionately influences the model.  
Next, we reshaped each image from 784 values into a 28 × 28 image. We did this because each row contains 784 points, each pertaining to a pixel in a 28x28 image. Reshaping the data allows us to display the images using Matplotlib before training the autoencoder.

### Pre-Encoding Visualizations

In [ ]:
plt.figure(figsize=(16,10))
for i in range(5):
    plt.subplot(1,5,i+1)
    plt.imshow(img_data[i],cmap="gray")
    plt.axis("off")

We plotted the first five images using using a grayscale colormap. These original images serve as the baseline we will compare the decoded images against when judging the autoencoder.  
Each image shows a light digit stroke on a dark background, roughly centred in the frame. Most pixels, especially around the edges, are 0 (black) and carry no information about the digit.  
Much of each image is "empty" space, which suggests the images can be compressed substantially without losing the digit itself.

### Reshape for Data Encoding

In [ ]:
img_data = img_data.reshape((-1,28**2))
print("Shape of Array:",img_data.shape)
print("Shape of Each Row:",img_data[0].shape)

We flattened each 28×28 image back into a one-dimensional array of 784 values. Keras `Dense` layers take one-dimensional input, so the autoencoder's input layer expects 784 values per image.  
The data shape is back to **(9998, 784)**, with values between 0 and 1, which is ready to feed into the network. 

In [ ]:
# Dummy Cell - Used for Run All Above - Remove Before Submission

## Model Selection

In [ ]:

def compile_autoencoder(data,autoencoder,weights,validation=None,optimizer="adadelta",loss="binary_crossentropy",epochs=50):
    if weights != None:
        autoencoder["decoded"].set_weights(weights)
    
    autoencoder["decoded"].compile(optimizer=optimizer,loss=loss)
    autoencoder["decoded"].fit(data,data,validation_data=validation,epochs=epochs)

def create_autoencoder(data,activation_function="relu",activation_final="sigmoid",loss_function="binary_crossentropy",optimizer="adadelta",weights = None, epochs=50):
    autoencoder = dict()
    train_data, test_data = tts(data,train_size=0.9,test_size=0.1,random_state=42)
    # Define Input Layer
    input_layer = Input(shape=(784,))
    # Define Encoding Layers
    hidden_encoding1 = Dense(1024,activation=act)(input_layer)
    hidden_encoding2 = Dense(512,activation=act)(hidden_encoding1)
    hidden_encoding3 = Dense(256,activation=act)(hidden_encoding2)
    hidden_encoding4 = Dense(128,activation=act)(hidden_encoding3)
    hidden_encoding5 = Dense(64,activation=act)(hidden_encoding4)
    hidden_encoding6 = Dense(32,activation=act)(hidden_encoding5)
    # Final Encoding Layer
    encoding_stage = Dense(16,activation=act)(hidden_encoding6)
    # Define Decoding Layers
    hidden_decoding1 = Dense(32,activation=act)(encoding_stage)
    hidden_decoding2 = Dense(64,activation=act)(hidden_decoding1)
    hidden_decoding3 = Dense(128,activation=act)(hidden_decoding2)
    hidden_decoding4 = Dense(256,activation=act)(hidden_decoding3)
    hidden_decoding5 = Dense(512,activation=act)(hidden_decoding4)
    hidden_decoding6 = Dense(1024,activation=act)(hidden_decoding5)
    # Final Decoding Layer
    decoding_stage = Dense(784,activation=act_fnl)(hidden_decoding6)
    # Define Autoencoder Model
    autoencoder = {"encoded": Model(input_layer,encoding_stage),
                    "decoded": Model(input_layer,decoding_stage),
                    "Final Activation": activation_final,
                    "Hidden Activation": activation_function,
                    "Loss Function": loss_function,
                    "Optimizer": optimizer,
    }
    # Set Starting Weights
    initial_weights = weights
        
    # Compile and Train Model
    compile_autoencoder(train_data,autoencoder,initial_weights,optimizer=optimizer,loss=loss,epochs=epochs)
    autoencoder["Loss"] = autoencoder["decoded"].evaluate(test_data,test_data,verbose=0)
    return autoencoder

def plot_images(data, autoencoders,num_img=5):
    data = data.reshape((-1,28**2))
    for key in autoencoders:
        encoded_img = autoencoders[key]["encoded"].predict(data[:num_img]).reshape((-1,4,4))
        decoded_img = autoencoders[key]["decoded"].predict(data[:num_img]).reshape((-1,28,28))
        loss = autoencoders[key]["Loss Function"]
        hidden_act = autoencoders[key]["Hidden Activation"]
        final_act = autoencoders[key]["Final Activation"]
        data = data.reshape((-1,28,28))
        # print("Autoencoder ID:",i)
        plt.figure(figsize=(16,10))
        for j in range(num_img):
            plt.subplot(3,num_img,j+1)
            plt.imshow(data[j],cmap="gray")
            plt.axis("off")

            plt.subplot(3,num_img,j+1+num_img)
            plt.imshow(encoded_img[j],cmap="gray")
            plt.axis("off")

            plt.subplot(3,num_img,j+1+num_img*2)
            plt.imshow(decoded_img[j],cmap="gray")
            plt.axis("off")
        data = data.reshape((-1,28**2))

In [ ]:
activation_function = ["relu"]
activation_final = ["sigmoid","softmax"]
loss_functions = ["binary_crossentropy","mean_squared_error"]
optimizer = ["adam","adadelta"]
epochs = [100]
autoencoders3 = dict()
i = 0
for act in activation_function:
    for act_fnl in activation_final:
        for loss in loss_functions:
            for opti in optimizer:
                for epoch in epochs:
                    if (act_fnl == "softmax" and loss != "binary_crossentropy" and opti !="binary_crossentropy"):
                        continue
                    elif (act_fnl == "sigmoid" and opti == "adadelta" and loss != "binary_crossentropy"):
                        continue
                    if i != 0:
                        w = autoencoders3[0]["decoded"].get_weights()
                    else:
                        w = None
                    autoencoders3[i] = create_autoencoder(img_data,act,act_fnl,loss,opti,w,epoch)
                    i += 1
df_encoded_pars3 = pd.DataFrame(autoencoders3).transpose().drop(columns=["decoded","encoded"])

In [ ]:
plot_images(img_data,autoencoders3)